In [ ]:
!wget http://nlp.stanford.edu/data/glove.6B.zip
!unzip -q glove.6B.zip

MAX_WORDS = 20000
MAX_LEN = 40   # slightly increased

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(df['final_text'])

X_seq = tokenizer.texts_to_sequences(df['final_text'])
X_pad = pad_sequences(X_seq, maxlen=MAX_LEN, padding='post')

X_train, X_val, y_train, y_val = train_test_split(
    X_pad, y, test_size=0.2, random_state=42
)

In [ ]:
embeddings_index = {}

with open("glove.6B.100d.txt", encoding="utf8") as f:
    for line in f:
        values = line.split()
        word = values[0]
        vec = np.asarray(values[1:], dtype='float32')
        embeddings_index[word] = vec

# Creating Embedding Metrix

EMBED_DIM = 100
word_index = tokenizer.word_index

embedding_matrix = np.zeros((MAX_WORDS, EMBED_DIM))

for word, i in word_index.items():
    if i < MAX_WORDS:
        vec = embeddings_index.get(word)
        if vec is not None:
            embedding_matrix[i] = vec

# BI-LSTM Model

model = Sequential()

model.add(Embedding(
    input_dim=MAX_WORDS,
    output_dim=EMBED_DIM,
    weights=[embedding_matrix],
    input_length=MAX_LEN,
    trainable=False   # Important: freeze embeddings
))

model.add(Bidirectional(LSTM(128, return_sequences=True)))
model.add(Bidirectional(LSTM(64)))

model.add(Dropout(0.5))

model.add(Dense(64, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

In [ ]:
#compile the model
model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

In [ ]:
model.build(input_shape=(None, MAX_LEN))
model.summary()

In [ ]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=8,
    batch_size=64
)

In [ ]:
#prediction

y_pred = model.predict(X_val).flatten()

from sklearn.metrics import mean_squared_error
import numpy as np

rmse = np.sqrt(mean_squared_error(y_val, y_pred))
print("\n BiLSTM + GloVe RMSE:", rmse)